CASA INTELIGENTE

Escribir las librerias

In [5]:
%%writefile -a C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp
#include <Arduino.h>
#include <Wire.h>
#include <DHT.h>
#include <BH1750.h>
#include <WiFiS3.h>
#include <WiFiUdp.h>
#include <string.h>

Appending to C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp


Configuración del entorno Wi-Fi, escribir el nombre del punto de acceso y la contraseña.

In [6]:
%%writefile -a C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp

//configuracion de red
char ssid[] = "OPPO A80 5G";
char password[] = "123456789";

WiFiServer servidor(80);

bool servidorIniciado = false;

unsigned long ultimoIntentoWiFi = 0;

Appending to C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp


Reloj y ajuste de la hora actual

In [7]:
%%writefile -a C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp

//reloj NTP
WiFiUDP udpNTP;
const unsigned int puertoNTP = 2390;
const long diferenciaHorario = -6L * 3600L;

unsigned long epochSincronizado = 0;
unsigned long millisSincronizado = 0;
unsigned long ultimoIntentoNTP = 0;

bool horaDisponible = false;
bool programacionActiva = false;

//horarios predeterminados, se pueden cambiar desde androidd
int horaDormir = 23;
int minutoDormir = 0;
int horaDespertar = 7;
int minutoDespertar = 0;

long ultimoMinutoRevisado = -1;

Appending to C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp


Configuración de pines y sensores.

In [8]:
%%writefile -a C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp

#define DHTPIN 2
#define DHTTYPE DHT11

#define VENTILADOR 5
#define ILUMINACION 6
#define SENSOR_PRESENCIA 7

DHT dht(DHTPIN, DHTTYPE);
BH1750 sensorLuz;

Appending to C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp


Variables globales

In [9]:
%%writefile -a C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp

//temperatura y humedad
float temperatura = 0;
float humedad = 0;
int pwmVentilador = 0;
int objetivoVentilador = 0;

//iluminacion
float lux = 0;
int pwmLuz = 0;
int objetivoLuz = 0;

//presencia
bool sistemaActivo = true;
bool confirmacionPendiente = false;
bool mantenerActivo = false;
bool modoDormir = false;
unsigned long ultimoMovimiento = 0;
const unsigned long tiempoAviso = 20000UL;
const unsigned long tiempoAusencia = 30000UL;

Appending to C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp


Tiempos de control

In [10]:
%%writefile -a C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp

unsigned long tiempoDHT = 0;
unsigned long tiempoLuz = 0;
unsigned long tiempoPWM = 0;
unsigned long ultimoReporte = 0;
const unsigned long intervaloDHT = 2000;
const unsigned long intervaloLuz = 200;
const unsigned long intervaloPWM = 30;

Appending to C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp


Funciones difusas del ventilador

In [11]:
%%writefile -a C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp

float fria(float x) {
    if (x <= 18) return 1;
    if (x >= 23) return 0;

    return (23 - x) / 5.0;
}
float templada(float x) {
    if (x <= 20 || x >= 30) return 0;

    if (x <= 25) {
        return (x - 20) / 5.0;
    }

    return (30 - x) / 5.0;
}
float calida(float x) {
    if (x <= 27 || x >= 37) return 0;

    if (x <= 32) {
        return (x - 27) / 5.0;
    }

    return (37 - x) / 5.0;
}
float caliente(float x) {
    if (x <= 34) return 0;
    if (x >= 38) return 1;

    return (x - 34) / 4.0;
}

Appending to C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp


Control difuso del ventilador

In [12]:
%%writefile -a C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp

void controlarVentilador(float temp) {

    float r1 = fria(temp);
    float r2 = templada(temp);
    float r3 = calida(temp);
    float r4 = caliente(temp);

    float s1 = 0;
    float s2 = 120;
    float s3 = 190;
    float s4 = 255;

    float suma = (r1 * s1) + (r2 * s2) + (r3 * s3) + (r4 * s4);

    float total = r1 + r2 + r3 + r4;

    if (total > 0) {
        objetivoVentilador = suma / total;
    }
    objetivoVentilador = constrain(
        objetivoVentilador, 0, 255
    );
    if (objetivoVentilador > 0 &&
        objetivoVentilador < 100) {

        objetivoVentilador = 100;
    }
}

Appending to C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp


Funciones difusas de la ilumincación

In [13]:
%%writefile -a C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp

float muyOscuro(float x) {
    if (x <= 2) return 1;
    if (x >= 15) return 0;

    return (15 - x) / 13.0;
}
float oscuro(float x) {
    if (x <= 8 || x >= 32) return 0;

    if (x <= 20) {
        return (x - 8) / 12.0;
    }

    return (32 - x) / 12.0;
}
float iluminado(float x) {
    if (x <= 25) return 0;
    if (x >= 40) return 1;

    return (x - 25) / 15.0;
}

Appending to C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp


Control difuso de la iluminación

In [15]:
%%writefile -a C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp

void controlarLuz(float iluminacion) {

    float r1 = muyOscuro(iluminacion);
    float r2 = oscuro(iluminacion);
    float r3 = iluminado(iluminacion);

    float s1 = 255;
    float s2 = 130;
    float s3 = 0;

    float suma = (r1 * s1) + (r2 * s2) + (r3 * s3);

    float total = r1 + r2 + r3;

    if (total > 0) {
        objetivoLuz = suma / total;
    }

    objetivoLuz = constrain(
        objetivoLuz, 0, 255
    );
}
void activarModoDormir();
void despertarSistema();

Appending to C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp


Funciones de reloj y programación

In [16]:
%%writefile -a C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp

String formarHora(int hora, int minuto) {
    String resultado = "";
    if (hora < 10) resultado += "0";
    resultado += String(hora);
    resultado += ":";
    if (minuto < 10) resultado += "0";
    resultado += String(minuto);
    return resultado;
}
bool convertirHora(String valor, int &hora, int &minuto) {
    valor.trim();

    if (valor.length() != 5 || valor.charAt(2) != ':') return false;

    if (!isDigit(valor.charAt(0)) || !isDigit(valor.charAt(1)) || !isDigit(valor.charAt(3)) || !isDigit(valor.charAt(4))) {
        return false;
    }

    hora = valor.substring(0, 2).toInt();
    minuto = valor.substring(3, 5).toInt();

    return hora >= 0 && hora <= 23 && minuto >= 0 && minuto <= 59;
}
unsigned long obtenerHoraUTC() {
    if (!horaDisponible) return 0;
    return epochSincronizado + (millis() - millisSincronizado) / 1000UL;
}
unsigned long obtenerHoraLocal() {
    long horaLocal = (long)obtenerHoraUTC() + diferenciaHorario;
    return (unsigned long)horaLocal;
}
void sincronizarHoraNTP() {
    if (WiFi.status() != WL_CONNECTED) return;

    Serial.println("SINCRONIZANDO RELOJ NTP...");

    IPAddress direccionNTP;
    if (WiFi.hostByName("pool.ntp.org", direccionNTP) != 1) {
        Serial.println("No se pudo localizar el servidor NTP");
        ultimoIntentoNTP = millis();
        return;
    }

    byte paqueteNTP[48];
    memset(paqueteNTP, 0, 48);

    paqueteNTP[0] = 0b11100011;
    paqueteNTP[1] = 0;
    paqueteNTP[2] = 6;
    paqueteNTP[3] = 0xEC;
    paqueteNTP[12] = 49;
    paqueteNTP[13] = 0x4E;
    paqueteNTP[14] = 49;
    paqueteNTP[15] = 52;

    udpNTP.stop();
    udpNTP.begin(puertoNTP);
    udpNTP.beginPacket(direccionNTP, 123);
    udpNTP.write(paqueteNTP, 48);
    udpNTP.endPacket();

    unsigned long inicio = millis();

    while (millis() - inicio < 1800) {
        int tamano = udpNTP.parsePacket();

        if (tamano >= 48) {
            udpNTP.read(paqueteNTP, 48);

            unsigned long segundos1900 = ((unsigned long)paqueteNTP[40] << 24) | ((unsigned long)paqueteNTP[41] << 16) | ((unsigned long)paqueteNTP[42] << 8) | (unsigned long)paqueteNTP[43];

            epochSincronizado = segundos1900 - 2208988800UL;
            millisSincronizado = millis();
            horaDisponible = true;

            unsigned long segundosDia = obtenerHoraLocal() % 86400UL;
            Serial.print("HORA LOCAL: ");
            Serial.println(formarHora(segundosDia / 3600UL, (segundosDia % 3600UL) / 60UL));
            ultimoIntentoNTP = millis();
            return;
        }
        delay(10);
    }
    Serial.println("No se recibio respuesta NTP");
    ultimoIntentoNTP = millis();
}
void revisarProgramacion() {
    if (!programacionActiva || !horaDisponible) return;

    unsigned long horaLocal = obtenerHoraLocal();
    unsigned long minutoActual = horaLocal / 60UL;

    if ((long)minutoActual == ultimoMinutoRevisado) return;
    ultimoMinutoRevisado = minutoActual;

    unsigned long segundosDia = horaLocal % 86400UL;
    int horaActual = segundosDia / 3600UL;
    int minutoActualDia = (segundosDia % 3600UL) / 60UL;

    if (horaActual == horaDormir && minutoActualDia == minutoDormir) {
        if (!modoDormir) {
            Serial.println("HORARIO AUTOMATICO: A DORMIR");
            activarModoDormir();
        }
    }

    if (horaActual == horaDespertar && minutoActualDia == minutoDespertar) {
        if (modoDormir) {
            Serial.println("HORARIO AUTOMATICO: A DESPERTAR");
            despertarSistema();
        }
    }
}


Appending to C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp


Suspender sistema

In [17]:
%%writefile -a C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp

void suspenderSistema() {
    sistemaActivo = false;
    confirmacionPendiente = false;

    objetivoVentilador = 0;
    objetivoLuz = 0;

    pwmVentilador = 0;
    pwmLuz = 0;

    analogWrite(VENTILADOR, 0);
    analogWrite(ILUMINACION, 0);

    Serial.println();
    Serial.println("SISTEMA SUSPENDIDO");
}

Appending to C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp


Reactivar sistema

In [18]:
%%writefile -a C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp

void reactivarSistema() {
    sistemaActivo = true;
    confirmacionPendiente = false;

    ultimoMovimiento = millis();

    udpNTP.begin(puertoNTP);

    Serial.println();
    Serial.println("SISTEMA REACTIVADO");
}

Appending to C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp


Modo dormir y despertar

In [19]:
%%writefile -a C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp

void activarModoDormir() {
    modoDormir = true;
    mantenerActivo = false;

    suspenderSistema();

    Serial.println("MODO DORMIR ACTIVADO");
    Serial.println("EL SENSOR PIR NO PUEDE DESPERTAR EL SISTEMA");
}
void despertarSistema() {
    modoDormir = false;
    mantenerActivo = false;

    reactivarSistema();

    Serial.println("MODO DORMIR DESACTIVADO");
}

Appending to C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp


Control de presencia

In [20]:
%%writefile -a C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp

void controlarPresencia() {
    //enn modo dormir, el movimiento no reactiva el sistema.
    if (modoDormir) {
        return;
    }

    int estadoPIR = digitalRead(SENSOR_PRESENCIA);

    if (estadoPIR == HIGH) {
        ultimoMovimiento = millis();

        if (!sistemaActivo) {
            reactivarSistema();
        }

        confirmacionPendiente = false;
    }

    //si el usuario mantiene el sistema activo, no se realiza la cuenta regresiva
    if (sistemaActivo && !mantenerActivo) {
        unsigned long ausencia =
            millis() - ultimoMovimiento;

        if (ausencia >= tiempoAviso &&
            ausencia < tiempoAusencia) {

            if (!confirmacionPendiente) {

                confirmacionPendiente = true;

                Serial.println();
                Serial.println("CONFIRMACION PENDIENTE");
                Serial.println("CONSULTAR APLICACION");
            }
        }
        if (ausencia >= tiempoAusencia) {

            suspenderSistema();

            Serial.println("TIEMPO DE AUSENCIA AGOTADO");
            Serial.println("VENTILADOR APAGADO");
            Serial.println("ILUMINACION APAGADA");
        }
    }
}

Appending to C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp


Actualizar salidas

In [21]:
%%writefile -a C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp

void actualizarSalidas() {
    if (pwmVentilador < objetivoVentilador) {

        pwmVentilador += 5;

        if (pwmVentilador > objetivoVentilador) {
            pwmVentilador = objetivoVentilador;
        }
    }
    else if (pwmVentilador > objetivoVentilador) {

        pwmVentilador -= 5;

        if (pwmVentilador < objetivoVentilador) {
            pwmVentilador = objetivoVentilador;
        }
    }
    if (pwmLuz > objetivoLuz) {

        pwmLuz -= 10;

        if (pwmLuz < objetivoLuz) {
            pwmLuz = objetivoLuz;
        }
    }
    else if (pwmLuz < objetivoLuz) {

        pwmLuz += 2;

        if (pwmLuz > objetivoLuz) {
            pwmLuz = objetivoLuz;
        }
    }

    analogWrite(VENTILADOR, pwmVentilador);
    analogWrite(ILUMINACION, pwmLuz);
}

Appending to C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp


Generar JSON para android con los datos del sistema

In [22]:
%%writefile -a C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp

String generarJSON() {
    bool movimiento = digitalRead(SENSOR_PRESENCIA) == HIGH;

    unsigned long ausencia = millis() - ultimoMovimiento;
    unsigned long restante = 0;

    if (sistemaActivo && ausencia < tiempoAusencia) {
        restante = (tiempoAusencia - ausencia) / 1000;
    }

    String horaActualTexto = "--:--";

    if (horaDisponible) {
        unsigned long horaLocal = obtenerHoraLocal();
        unsigned long segundosDia = horaLocal % 86400UL;
        horaActualTexto = formarHora(segundosDia / 3600UL, (segundosDia % 3600UL) / 60UL);
    }

    String json = "{";

    json += "\"temperatura\":";
    json += String(temperatura, 2);

    json += ",\"humedad\":";
    json += String(humedad, 2);

    json += ",\"iluminacion\":";
    json += String(lux, 2);

    json += ",\"movimiento\":";
    json += movimiento ? "true" : "false";

    json += ",\"presencia\":";
    json += movimiento ? "true" : "false";

    json += ",\"ventiladorPWM\":";
    json += String(pwmVentilador);

    json += ",\"lucesPWM\":";
    json += String(pwmLuz);

    json += ",\"estadoSistema\":\"";

    if (modoDormir) {
        json += "DURMIENDO";
    }
    else if (sistemaActivo) {
        json += "ACTIVO";
    }
    else {
        json += "SUSPENDIDO";
    }

    json += "\"";

    json += ",\"modoDormir\":";
    json += modoDormir ? "true" : "false";

    json += ",\"confirmacionPendiente\":";
    json += confirmacionPendiente ? "true" : "false";

    json += ",\"tiempoRestante\":";
    json += String(restante);

    json += ",\"horaActual\":\"";
    json += horaActualTexto;
    json += "\"";

    json += ",\"horaDormir\":\"";
    json += formarHora(horaDormir, minutoDormir);
    json += "\"";

    json += ",\"horaDespertar\":\"";
    json += formarHora(horaDespertar, minutoDespertar);
    json += "\"";

    json += ",\"programacionActiva\":";
    json += programacionActiva ? "true" : "false";

    json += ",\"horaSincronizada\":";
    json += horaDisponible ? "true" : "false";

    json += "}";

    return json;
}

Appending to C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp


Respuesta HTTP

In [23]:
%%writefile -a C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp

void responderCliente(WiFiClient &cliente, String contenido) {

    cliente.println("HTTP/1.1 200 OK");
    cliente.println("Content-Type: application/json; charset=utf-8");
    cliente.println("Access-Control-Allow-Origin: *");
    cliente.println("Connection: close");
    cliente.println();
    cliente.println(contenido);
}

Appending to C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp


Atender peticiones de android

In [25]:
%%writefile -a C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp

void atenderCliente(){
    WiFiClient cliente = servidor.available();

    if (!cliente){
        return;
    }

    unsigned long inicio = millis();

    while (!cliente.available() && millis() - inicio < 1000){
        delay(1);
    }

    if (!cliente.available()){
        cliente.stop();
        return;
    }

    String peticion = cliente.readStringUntil('\r');

    Serial.println();
    Serial.print("Peticion: ");
    Serial.println(peticion);

    # //leer encabezados HTTP
    while (cliente.connected()){
        String linea = cliente.readStringUntil('\n');

        if (linea.length() <= 1){
            break;
        }
    }

    //consultar sensores
    if (peticion.indexOf("GET /datos") >= 0){
        responderCliente(cliente, generarJSON());
    }

    //configurar horarios de android
    else if (peticion.indexOf("accion=PROGRAMAR") >= 0){

        int inicioDormir = peticion.indexOf("dormir=");
        int inicioDespertar = peticion.indexOf("despertar=");

        if (inicioDormir >= 0 && inicioDespertar >= 0){

            int finDormir = peticion.indexOf("&", inicioDormir);
            if (finDormir < 0) finDormir = peticion.indexOf(" ", inicioDormir);
            int finDespertar = peticion.indexOf(" ", inicioDespertar);

            String textoDormir = peticion.substring(inicioDormir + 7, finDormir);
            String textoDespertar = peticion.substring(inicioDespertar + 10, finDespertar);

            int nuevaHoraDormir, nuevoMinutoDormir;
            int nuevaHoraDespertar, nuevoMinutoDespertar;

            bool dormirValido = convertirHora(textoDormir, nuevaHoraDormir, nuevoMinutoDormir);
            bool despertarValido = convertirHora(textoDespertar, nuevaHoraDespertar, nuevoMinutoDespertar);

            if (dormirValido && despertarValido && (nuevaHoraDormir != nuevaHoraDespertar || nuevoMinutoDormir != nuevoMinutoDespertar)){
                horaDormir = nuevaHoraDormir;
                minutoDormir = nuevoMinutoDormir;
                horaDespertar = nuevaHoraDespertar;
                minutoDespertar = nuevoMinutoDespertar;
                ultimoMinutoRevisado = -1;

                responderCliente(cliente, "{\"mensaje\":\"Horarios configurados\"}");
                Serial.println("HORARIOS ACTUALIZADOS");
                Serial.print("Dormir: ");
                Serial.println(formarHora(horaDormir, minutoDormir));
                Serial.print("Despertar: ");
                Serial.println(formarHora(horaDespertar, minutoDespertar));
            }
            else{
                responderCliente(cliente, "{\"error\":\"Formato u horarios no validos\"}");
            }
        }
        else{
            responderCliente(cliente, "{\"error\":\"Faltan parametros dormir y despertar\"}");
        }
    }

    //activar programacion automatica
    else if (peticion.indexOf("accion=PROGRAMACION_ON") >= 0){
        programacionActiva = true;
        ultimoMinutoRevisado = -1;
        responderCliente(cliente, "{\"mensaje\":\"Programacion automatica activada\"}");
        Serial.println("PROGRAMACION AUTOMATICA ACTIVADA");
    }

    //desactivar programacion automatica
    else if (peticion.indexOf("accion=PROGRAMACION_OFF") >= 0){
        programacionActiva = false;
        responderCliente(cliente, "{\"mensaje\":\"Programacion automatica desactivada\"}");
        Serial.println("PROGRAMACION AUTOMATICA DESACTIVADA");
    }

    //activar modo dormir manualmente
    else if (peticion.indexOf("accion=DORMIR") >= 0){
        activarModoDormir();

        responderCliente(cliente,
            "{\"mensaje\":\"Modo dormir activado manualmente\"}");
    }

    //despertar manualmente
    else if (peticion.indexOf("accion=DESPERTAR") >= 0){
        despertarSistema();

        responderCliente(cliente,
            "{\"mensaje\":\"Sistema despertado manualmente\"}");
    }

    //mantener sistema activo
    else if (peticion.indexOf("accion=MANTENER") >= 0) {

        if (modoDormir){
            responderCliente(cliente,
                "{\"mensaje\":\"Primero debes despertar el sistema\"}");
        }
        else{
            mantenerActivo = true;
            sistemaActivo = true;
            confirmacionPendiente = false;

            responderCliente(cliente,
                "{\"mensaje\":\"Sistema activo permanentemente\"}");

            Serial.println();
            Serial.println("MODO MANTENER ACTIVADO");
            Serial.println("SUSPENSION AUTOMATICA DESHABILITADA");
        }
    }

    //suspender sistema
    else if (peticion.indexOf("accion=SUSPENDER") >= 0) {

        mantenerActivo = false;
        modoDormir = false;

        suspenderSistema();

        responderCliente(cliente,
            "{\"mensaje\":\"Sistema suspendido\"}");
    }

    else{

        responderCliente(cliente,
            "{\"mensaje\":\"Ruta no encontrada\"}");
    }

    delay(2);
    cliente.stop();
}

Appending to C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp


Conectar a la red Wi-Fi

In [27]:
%%writefile -a C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp

void conectarWiFi(){
    Serial.println();
    Serial.println("CONECTANDO A WIFI...");

    WiFi.begin(ssid, password);

    unsigned long inicio = millis();

    while (WiFi.status() != WL_CONNECTED && millis() - inicio < 20000) {
        Serial.print(".");
        delay(500);
    }

    Serial.println();

    if (WiFi.status() == WL_CONNECTED){
        Serial.println("WIFI CONECTADO");

        Serial.print("Direccion IP: ");
        Serial.println(WiFi.localIP());

        Serial.print("Intensidad de senal: ");
        Serial.print(WiFi.RSSI());
        Serial.println(" dBm");

        if (!servidorIniciado){
            servidor.begin();
            servidorIniciado = true;

            Serial.println("SERVIDOR HTTP INICIADO");
            Serial.println("Puerto: 80");
        }

        if (!horaDisponible || millis() - ultimoIntentoNTP >= 21600000UL){
            sincronizarHoraNTP();
        }
    }
    else{
        Serial.println("NO SE PUDO CONECTAR A WIFI");
    }

    ultimoIntentoWiFi = millis();
}

Appending to C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp


Configuración inicial

In [ ]:
%%writefile -a C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp

void setup(){

    Serial.begin(9600);

    Wire.begin();

    dht.begin();

    pinMode(VENTILADOR, OUTPUT);
    pinMode(ILUMINACION, OUTPUT);
    pinMode(SENSOR_PRESENCIA, INPUT);

    analogWrite(VENTILADOR, 0);
    analogWrite(ILUMINACION, 0);

    if (sensorLuz.begin()){
        Serial.println("BH1750 iniciado correctamente");
    }
    else{
        Serial.println("Error al iniciar BH1750");
    }

    ultimoMovimiento = millis();

    udpNTP.begin(puertoNTP);

    Serial.println();
    Serial.println("SISTEMA DIFUSO INICIADO");
    Serial.println("Tiempo de aviso: 20 segundos");
    Serial.println("Tiempo de suspension: 30 segundos");

    conectarWiFi();
}

Appending to C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp


Loop principal del sistema

In [31]:
%%writefile -a C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp

void loop(){

    unsigned long tiempoActual = millis();

    //Control de presencia y suspension automatica
    controlarPresencia();

    //Lectura de dht11
    if (tiempoActual - tiempoDHT >= intervaloDHT){
        tiempoDHT = tiempoActual;

        float nuevaTemperatura = dht.readTemperature();
        float nuevaHumedad = dht.readHumidity();

        if (!isnan(nuevaTemperatura)){
            temperatura = nuevaTemperatura;

            if (sistemaActivo){
                controlarVentilador(temperatura);
            }
        }
        else{
            Serial.println("Error leyendo DHT11");
        }
        if (!isnan(nuevaHumedad)){
            humedad = nuevaHumedad;
        }
    }

    //Lestura de sensor de luz BH1750
    if (tiempoActual - tiempoLuz >= intervaloLuz){
        tiempoLuz = tiempoActual;

        float nuevaLux = sensorLuz.readLightLevel();

        if (nuevaLux >= 0){
            lux = nuevaLux;

            if (sistemaActivo){
                controlarLuz(lux);
            }
        }
        else{
            Serial.println("Error leyendo BH1750");
        }
    }

    //proteccion de sistema suspendido

    if (!sistemaActivo){
        objetivoVentilador = 0;
        objetivoLuz = 0;

        pwmVentilador = 0;
        pwmLuz = 0;

        analogWrite(VENTILADOR, 0);
        analogWrite(ILUMINACION, 0);
    }

    //actualizar salidas
    if (tiempoActual - tiempoPWM >= intervaloPWM){
        tiempoPWM = tiempoActual;

        actualizarSalidas();
    }

    //atender aplicacion andoid
    if (WiFi.status() == WL_CONNECTED){
        atenderCliente();
    }
    else{
        if (servidorIniciado){
            servidorIniciado = false;

            Serial.println("CONEXION WIFI PERDIDA");
        }
        if (tiempoActual - ultimoIntentoWiFi >= 10000){
            conectarWiFi();
        }
    }

    //reloj y programacion automatica
    if (WiFi.status() == WL_CONNECTED){
        if ((!horaDisponible && tiempoActual - ultimoIntentoNTP >= 60000UL) || (horaDisponible && tiempoActual - ultimoIntentoNTP >= 21600000UL)){
            sincronizarHoraNTP();
        }
        revisarProgramacion();
    }

    //monitor serial
    if (tiempoActual - ultimoReporte >= 1000){
        ultimoReporte = tiempoActual;

        Serial.println();
        Serial.println("========= SISTEMA DIFUSO =========");

        Serial.print("Estado: ");

        if (modoDormir){
            Serial.println("DURMIENDO");
        }
        else if (sistemaActivo){
            Serial.println("ACTIVO");
        }
        else{
            Serial.println("SUSPENDIDO");
        }

        Serial.print("Temperatura: ");
        Serial.print(temperatura);
        Serial.println(" C");

        Serial.print("Humedad: ");
        Serial.print(humedad);
        Serial.println(" %");

        Serial.print("Ventilador PWM: ");
        Serial.println(pwmVentilador);

        Serial.print("Iluminacion: ");
        Serial.print(lux);
        Serial.println(" lux");

        Serial.print("Luces PWM: ");
        Serial.println(pwmLuz);

        Serial.print("Modo dormir: ");
        Serial.println(modoDormir ? "ACTIVADO" : "DESACTIVADO");

        Serial.print("Hora sincronizada: ");
        Serial.println(horaDisponible ? "SI" : "NO");

        Serial.print("Hora actual: ");
        if (horaDisponible){
            unsigned long segundosDia = obtenerHoraLocal() % 86400UL;
            Serial.println(formarHora(segundosDia / 3600UL, (segundosDia % 3600UL) / 60UL));
        }
        else{
            Serial.println("--:--");
        }

        Serial.print("Programacion automatica: ");
        Serial.println(programacionActiva ? "ACTIVA" : "DESACTIVADA");

        Serial.print("Horario dormir: ");
        Serial.println(formarHora(horaDormir, minutoDormir));

        Serial.print("Horario despertar: ");
        Serial.println(formarHora(horaDespertar, minutoDespertar));

        Serial.print("Confirmacion pendiente: ");
        Serial.println(confirmacionPendiente ? "SI" : "NO");

        unsigned long ausencia =
            millis() - ultimoMovimiento;

        unsigned long restante = 0;

        if (sistemaActivo && ausencia < tiempoAusencia){
            restante = (tiempoAusencia - ausencia) / 1000;
        }

        Serial.print("Tiempo para suspension: ");
        Serial.print(restante);
        Serial.println(" segundos");

        Serial.println("=================================");
    }
}

Appending to C:\Users\adanv\Documents\PlatformIO\Projects\Doc_R4\src\main.cpp
